<a href="https://colab.research.google.com/github/Learn-AIMLOps/Instruction_fine_tunning/blob/main/chalML_tune_model_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import userdata
hf_write_token = userdata.get("HF_WRITE_TOKEN")

In [2]:
# STEP 1 — Install Libraries

!pip install -q transformers accelerate peft bitsandbytes trl datasets huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 35.8 MB/s eta 0:00:00


In [3]:
# STEP 2 — Login to HuggingFace
from huggingface_hub import login
login(token=hf_write_token)

In [4]:
# STEP 3 — Load Base Model (4bit)

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "Qwen/Qwen2-1.5B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

In [5]:
# STEP 4 — Load Domain Adapter

from peft import PeftModel

domain_adapter_path = "RohitGu/Qwen2-1.5B-LLMOps-LoRA"

model_with_domain = PeftModel.from_pretrained(
    base_model,
    domain_adapter_path
)

adapter_config.json:   0%|          | 0.00/968 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/4.37M [00:00<?, ?B/s]

In [6]:
# STEP 5 — Merge Domain Adapter

merged_model = model_with_domain.merge_and_unload()

merged_model.save_pretrained("merged_domain_model")
tokenizer.save_pretrained("merged_domain_model")

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('merged_domain_model/tokenizer_config.json',
 'merged_domain_model/chat_template.jinja',
 'merged_domain_model/tokenizer.json')

In [7]:
# STEP 6 — Reload merged model in 4bit

base_model = AutoModelForCausalLM.from_pretrained(
    "merged_domain_model",
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:246: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [8]:
# STEP 7 — Load Instruction Adapter

stage4_adapter = "RohitGu/Qwen2-1.5B-LLMOps-Instruct-LoRA"

model = PeftModel.from_pretrained(
    base_model,
    stage4_adapter
)

adapter_config.json:   0%|          | 0.00/973 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/4.37M [00:00<?, ?B/s]

In [9]:
# STEP 8 — Pre Training Test
# It may respond but not perfectly role-aware.

def generate_response(model, tokenizer, messages):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        top_p=0.9
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

test_messages = [
    {"role": "system", "content": "You are helpful."},
    {"role": "user", "content": "Explain LLM in simple terms."}
]

print(generate_response(model, tokenizer, test_messages))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


system
You are helpful.
user
Explain LLM in simple terms.
assistant
LLMs (Language Model) are advanced artificial intelligence systems that can understand and generate human-like text. They are trained on vast amounts of text data, including books, articles, and web pages, to learn how to generate coherent and contextually relevant text. LLMs are particularly useful for tasks such as text summarization, question answering, and text generation. They are also used in natural language processing (NLP) applications such as language translation, speech recognition, and sentiment analysis. LLMs are a powerful tool for automating tasks that require human-level language understanding and generation. They are particularly useful for tasks that require a high degree of accuracy and consistency, such as legal document translation, medical record transcription, and financial reporting. LLMs are also being used in areas such as cybersecurity, fraud detection, and fraud prevention. They are particul

In [15]:
# STEP 9 — Load Alpaca Dataset
from datasets import load_dataset

dataset = load_dataset("RohitGu/llmops-guide-ift-datasets", split="train")

In [16]:
# STEP 10 — Synthetic Multi-Turn Conversation Generation ⭐ (IMPORTANT)
def create_synthetic_chat(example):
    topic = example["instruction"]

    improved_instruction = f"""
Create a short 3-turn technical conversation about:
{topic}

Rules:
- Short answers
- Clear explanation
- Conversational tone
"""

    example["instruction"] = improved_instruction.strip()
    return example


dataset = dataset.map(create_synthetic_chat)

In [17]:
print(dataset)

Dataset({
    features: ['instruction', 'input', 'response'],
    num_rows: 91
})


In [18]:
# STEP 11 — ChatML Formatting
# STEP 11 — Convert Dataset → ChatML Format (Correct Version)

def convert_to_chatml(example):

    user_content = example["instruction"]

    if example["input"] and example["input"].strip() != "":
        user_content += "\n" + example["input"]

    messages = [
        {"role": "system", "content": "You are a helpful AI assistant."},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": example["response"]}
    ]

    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": formatted_text}


chat_dataset = dataset.map(
    convert_to_chatml,
    remove_columns=dataset.column_names
)

Map:   0%|          | 0/91 [00:00<?, ? examples/s]

In [19]:
print(chat_dataset.column_names)
print(chat_dataset[0])

['text']
{'text': '<|im_start|>system\nYou are a helpful AI assistant.<|im_end|>\n<|im_start|>user\nCreate a short 3-turn technical conversation about:\nExplain why small language models offer a more resource-efficient alternative than larger ones.\n\nRules:\n- Short answers\n- Clear explanation\n- Conversational tone\nSmall language models are often used in conjunction with large-scale pre-trained models to scale up the computational resources required to train them.<|im_end|>\n<|im_start|>assistant\nSmall language models require less computational power and data to train compared to larger models, making them a more resource-efficient option.<|im_end|>\n'}


In [20]:
print(type(dataset))
print(dataset.column_names)

<class 'datasets.arrow_dataset.Dataset'>
['instruction', 'input', 'response']


In [21]:
# STEP 12 — Quality Filtering
chat_dataset = chat_dataset.filter(
    lambda x: len(x["text"]) < 1024
)

Filter:   0%|          | 0/91 [00:00<?, ? examples/s]

In [22]:
# Add NEW LoRA

from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj","k_proj","v_proj","o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


In [23]:
# STEP 13 — Training Configuration (Optimal for T4)
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir="./chat_adapter",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=5,
    learning_rate=5e-5,
    logging_steps=5,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    packing=False
)

In [24]:
# STEP 14 — Train Model
trainer = SFTTrainer(
    model=model,
    train_dataset=chat_dataset,
    formatting_func=lambda x: x["text"],
    args=sft_config
)

trainer.train()

Applying formatting function to train dataset:   0%|          | 0/68 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/68 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/68 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/68 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
5,2.911857
10,2.670642
15,2.473414
20,2.408365
25,2.388005


TrainOutput(global_step=25, training_loss=2.5704567337036135, metrics={'train_runtime': 296.0023, 'train_samples_per_second': 1.149, 'train_steps_per_second': 0.084, 'total_flos': 309493233208320.0, 'train_loss': 2.5704567337036135})

In [25]:
# STEP 15 — Save Model
trainer.model.save_pretrained("./chat_adapter")
tokenizer.save_pretrained("./chat_adapter")

('./chat_adapter/tokenizer_config.json',
 './chat_adapter/chat_template.jinja',
 './chat_adapter/tokenizer.json')

In [26]:
# STEP 16 — Upload to HuggingFace
trainer.model.push_to_hub("RohitGu/Qwen2-1.5B-LLMOps-chatMl-LoRA-v1", token=hf_write_token)
tokenizer.push_to_hub("RohitGu/Qwen2-1.5B-LLMOps-chatMl-LoRA-v1", token=hf_write_token)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  94%|#########4| 8.23MB / 8.75MB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp53a1a773/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

CommitInfo(commit_url='https://huggingface.co/RohitGu/Qwen2-1.5B-LLMOps-chatMl-LoRA-v1/commit/895f94134cfafb50c068ad29a499aadb29f183cb', commit_message='Upload tokenizer', commit_description='', oid='895f94134cfafb50c068ad29a499aadb29f183cb', pr_url=None, repo_url=RepoUrl('https://huggingface.co/RohitGu/Qwen2-1.5B-LLMOps-chatMl-LoRA-v1', endpoint='https://huggingface.co', repo_type='model', repo_id='RohitGu/Qwen2-1.5B-LLMOps-chatMl-LoRA-v1'), pr_revision=None, pr_num=None)

In [27]:
# STEP 17 — Evaluation Testing
def generate_response(prompt):

    messages = [
        {"role": "system", "content": "You are helpful."},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        temperature=0.3,
        top_p=0.8,
        repetition_penalty=1.2
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


print(generate_response("Explain LLM briefly."))

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


system
You are helpful.
user
Explain LLM briefly.
assistant
LL,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
